# 04 — Model Evaluation & Prediction

**职责**: 加载 Top 3 模型 → 残差分析 → 测试集预测 → 输出提交文件

**输入**: `outputs/models/*.pkl`, `outputs/models/selected_features.json`, `data/raw/train_data.csv`  
**输出**: `outputs/predictions/submission.csv`, `outputs/figures/evaluation_*.png`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

from config import DATA_RAW, DATA_TEST, TARGET, MODEL_DIR, PRED_DIR, FIG_DIR
from src.preprocessing import load_and_clean, build_feature_matrix
from src.models import load_model, list_runs

# ── 指定要评估的 run_id（None = 用最近一次） ──
EVAL_RUN_ID = None

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. 加载数据 & 模型

In [ ]:
all_runs = list_runs()
run_id = EVAL_RUN_ID or (all_runs[-1] if all_runs else None)
print(f"Available runs: {all_runs}")
print(f"Evaluating run: {run_id}\n")

run_dir = MODEL_DIR / run_id
with open(run_dir / "selected_features.json") as f:
    selected_features = json.load(f)
print(f"Selected features: {len(selected_features)}")

df = load_and_clean(DATA_RAW)
df = df.sort_values('startdate').reset_index(drop=True)
X_full, y_full, _ = build_feature_matrix(df, TARGET)

split_idx = int(len(X_full) * 0.8)
X_test = X_full.iloc[split_idx:][selected_features]
y_test = y_full.iloc[split_idx:]
print(f"Test set: {X_test.shape}")

top3_names = [p.stem for p in sorted(run_dir.glob("*.pkl"))]
models = {name: load_model(name, run_id=run_id) for name in top3_names}
print(f"Loaded models: {list(models.keys())}")

## 2. 模型对比评估

In [ ]:
eval_rows = []
predictions = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    eval_rows.append({'Model': name, 'Test RMSE': round(rmse, 4), 'Test R²': round(r2, 4)})
    predictions[name] = y_pred
    print(f"[{name}] RMSE={rmse:.4f}, R²={r2:.4f}")

eval_df = pd.DataFrame(eval_rows).set_index('Model').sort_values('Test RMSE')
best_name = eval_df.index[0]
print(f"\nBest: {best_name}")
eval_df

## 3. 残差分析

对 Top 3 中表现最好的模型做残差图，检查预测偏差。

In [ ]:
best_model = models[best_name]
y_pred = predictions[best_name]
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Residual Analysis — {best_name}", fontsize=16, fontweight='bold')

ax = axes[0]
ax.scatter(y_pred, residuals, alpha=0.08, s=3, color='steelblue')
ax.axhline(0, color='crimson', linestyle='--')
ax.set_xlabel('Predicted (°C)')
ax.set_ylabel('Residual (°C)')
ax.set_title('Residuals vs Predicted')

ax = axes[1]
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white')
ax.axvline(0, color='crimson', linestyle='--')
ax.axvline(residuals.mean(), color='orange', linestyle='--',
           label=f'Mean={residuals.mean():.3f}')
ax.set_xlabel('Residual (°C)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')
ax.legend()

FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(FIG_DIR / 'evaluation_residuals.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Saved → {FIG_DIR / 'evaluation_residuals.png'}")

## 4. 测试集预测 & 提交

In [ ]:
# TODO: 确认测试集文件存在后取消注释
# df_test = load_and_clean(DATA_TEST)
# df_test = df_test.sort_values('startdate').reset_index(drop=True)
# X_submit, _, _ = build_feature_matrix(df_test, TARGET)
# X_submit = X_submit[selected_features]
#
# submit_pred = best_model.predict(X_submit)
#
# PRED_DIR.mkdir(parents=True, exist_ok=True)
# submission = pd.DataFrame({'id': df_test.index, TARGET: submit_pred})
# submission.to_csv(PRED_DIR / 'submission.csv', index=False)
# print(f"Saved → {PRED_DIR / 'submission.csv'}")
# submission.head()